# Виртуальное окружение и Git: первые коммиты

Занятие 17. Python, 9 класс (углублённый). Раздел 3.1 (окончание) и раздел 3.2, пара 1.

**План пары**

1. Виртуальное окружение
2. Зависимости проекта: `requirements.txt`
3. Git: репозиторий и первый коммит
4. Изменения: `status`, `diff` и индекс
5. История и отмена изменений
6. Что не хранить в репозитории: `.gitignore`

На прошлой паре мы приводили в порядок код внутри файла. Сегодня речь пойдёт о том, что вокруг кода.
Первая половина пары о библиотеках: как дать проекту нужные ему версии и как повторить их на другом
компьютере. Вторая половина о Git — программе, которая хранит историю изменений проекта.

Всю пару будем работать с маленьким проектом «Шахматный турнир»: программа читает результаты партий
из файла и печатает турнирную таблицу.

`make_project`, `show` и `run` те же, что на прошлых парах: создать файлы проекта, показать файл,
запустить `python`. Новые инструменты: `write(путь, текст)` переписывает один файл и не трогает
остальные, `remove(путь)` удаляет файл или папку, `show_tree()` показывает, какие файлы есть в проекте,
`venv_run(...)` запускает Python виртуального окружения, `git("...")` выполняет команду Git в папке
проекта. В блоке 2 нужен интернет: будем ставить библиотеку из PyPI. Git должен быть установлен,
в Colab он уже есть. Запустите служебную ячейку один раз.

In [ ]:
# Служебная ячейка: запустите один раз. Разбираться в ней не нужно.
import os
import re
import shlex
import shutil
import stat
import subprocess
import sys
import tempfile
import textwrap
from datetime import datetime, timedelta, timezone
from pathlib import Path

sys.dont_write_bytecode = True        # чтобы после правки файла Python не взял его старую версию
_BASE = Path(tempfile.gettempdir()) / "py9_17_lecture"
_BASE.mkdir(parents=True, exist_ok=True)
ROOT = _BASE / "project"
_WINDOWS = os.name == "nt"
_START = datetime(2026, 11, 9, 10, 0, tzinfo=timezone(timedelta(hours=3)))
_ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
_ENV = {name: value for name, value in os.environ.items()
        if not name.startswith(("GIT_", "PIP_"))
        and name not in ("VIRTUAL_ENV", "FORCE_COLOR", "CLICOLOR_FORCE", "PY_COLORS")}
_ENV.update({
    "PYTHONDONTWRITEBYTECODE": "1", "PYTHONIOENCODING": "utf-8", "PYTHON_COLORS": "0",
    "NO_COLOR": "1", "TERM": "dumb", "LANGUAGE": "en_US:en",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PIP_NO_INPUT": "1", "PIP_ROOT_USER_ACTION": "ignore",
    "PIP_PROGRESS_BAR": "off",
    # общие настройки Git — во временном файле, ваши настоящие настройки не меняются
    "GIT_CONFIG_GLOBAL": str(_BASE / "gitconfig"), "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CEILING_DIRECTORIES": str(_BASE), "GIT_PAGER": "cat", "PAGER": "cat",
    "GIT_EDITOR": "true", "GIT_TERMINAL_PROMPT": "0",
})


def _rmtree(path):
    def retry(func, name, *_):
        os.chmod(name, stat.S_IWRITE)         # в Windows файлы внутри .git только для чтения
        func(name)
    if sys.version_info >= (3, 12):
        shutil.rmtree(path, onexc=retry)
    else:
        shutil.rmtree(path, onerror=retry)


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8", newline="\n")


def make_project(files):
    """Создаёт папку проекта: files — словарь «путь -> текст файла». Прошлая версия стирается."""
    if ROOT.exists():
        _rmtree(ROOT)
    ROOT.mkdir(parents=True)
    for name, text in files.items():
        _write(ROOT / name, text)


def write(path, text):
    """Записывает один файл проекта, остальные файлы не трогает."""
    _write(ROOT / path, text)


def remove(path):
    """Удаляет файл или папку проекта."""
    target = ROOT / path
    if target.is_dir() and not target.is_symlink():
        _rmtree(target)
    elif target.exists() or target.is_symlink():
        target.unlink()


def show(path):
    """Печатает файл проекта с номерами строк."""
    print(f"----- {path} -----")
    for number, line in enumerate((ROOT / path).read_text(encoding="utf-8").splitlines(), start=1):
        print(f"{number:3d}  {line}")


def show_tree(path=".", depth=None):
    """Показывает файлы и папки проекта; depth — сколько уровней. Содержимое .git не показывается."""
    start = ROOT / path
    print("project/" if path == "." else f"{path}/")

    def walk(folder, level):
        for item in sorted(folder.iterdir(), key=lambda item: item.name):
            folder_like = item.is_dir() and not item.is_symlink()
            print("    " * level + item.name + ("/" if item.is_dir() else ""))
            if folder_like and item.name != ".git" and (depth is None or level < depth):
                walk(item, level + 1)

    walk(start, 1)


def _execute(command, env=None):
    result = subprocess.run(command, cwd=ROOT, env=env or _ENV, capture_output=True, text=True,
                            encoding="utf-8", errors="replace", stdin=subprocess.DEVNULL, timeout=600)
    return result.returncode, _ANSI.sub("", result.stdout), _ANSI.sub("", result.stderr).strip()


def _report(code, out, error):
    print(out, end="")
    if code != 0 and "Traceback" in error:    # от трассировки оставляем последнюю строку
        error = error.splitlines()[-1]
    if error:
        print("[ошибка]" if code != 0 else "", error)


def _make_venv(args):
    # python -m venv ...; если в системе нет ensurepip (так бывает в Colab), окружение делает virtualenv
    code, out, error = _execute([sys.executable, *args])
    if code != 0 and "ensurepip" in error:
        for name in args[2:]:
            if not name.startswith("-") and (ROOT / name).exists():
                _rmtree(ROOT / name)
        _execute([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        options = ["--no-pip" if arg == "--without-pip" else arg for arg in args[2:]]
        code, out, error = _execute([sys.executable, "-m", "virtualenv", "-q", *options])
        if code == 0:
            out += ("(в этой системе у venv нет модуля ensurepip, поэтому окружение "
                    "создано пакетом virtualenv — результат тот же)\n")
        else:
            error = ("в этой системе у venv нет модуля ensurepip, а поставить virtualenv "
                     "не получилось — проверьте интернет")
    return code, out, error


def run(*args):
    """Запускает в папке проекта  python args...  — тот Python, на котором работает ноутбук."""
    print("$ python", *args)
    ROOT.mkdir(parents=True, exist_ok=True)
    if args[:2] == ("-m", "venv"):
        _report(*_make_venv(args))
    else:
        _report(*_execute([sys.executable, *args]))


def _venv_python(venv):
    return ROOT / venv / ("Scripts/python.exe" if _WINDOWS else "bin/python")


def venv_run(*args, venv=".venv"):
    """Запускает в папке проекта Python виртуального окружения (по умолчанию из папки .venv)."""
    print("$", f"{venv}\\Scripts\\python" if _WINDOWS else f"{venv}/bin/python", *args)
    python = _venv_python(venv)
    if not python.exists():
        print(f"[ошибка] окружения {venv} нет, сначала создайте его: run('-m', 'venv', '{venv}')")
        return
    env = {name: value for name, value in _ENV.items() if name not in ("PYTHONPATH", "PYTHONHOME")}
    _report(*_execute([str(python), *args], env))


def _git(args, env):
    return subprocess.run(["git", "-c", "core.quotePath=false", "-c", "color.ui=never", *args],
                          cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, text=True, encoding="utf-8", errors="replace",
                          timeout=60)


def git(command):
    """Выполняет в папке проекта команду  git command  и печатает, что она вывела."""
    print("$ git", command)
    if shutil.which("git") is None:
        print("[ошибка] Git не установлен: скачайте его с https://git-scm.com и перезапустите Jupyter")
        return
    ROOT.mkdir(parents=True, exist_ok=True)
    # время коммита зависит только от их числа — поэтому номера коммитов у всех одинаковые
    count = _git(["rev-list", "--count", "HEAD"], _ENV).stdout.strip()
    moment = _START + timedelta(minutes=5 * (int(count) if count.isdigit() else 0))
    stamp = f"{int(moment.timestamp())} +0300"
    env = {**_ENV, "GIT_AUTHOR_DATE": stamp, "GIT_COMMITTER_DATE": stamp}
    text = _ANSI.sub("", _git(shlex.split(command), env).stdout).rstrip()
    if text:
        print(text)


if shutil.which("git") is None:
    print("Внимание: Git не найден. Установите его с https://git-scm.com, блоки 3–6 без него не работают.")
print("Служебная ячейка: ок")

Служебная ячейка: ок


## 1. Виртуальное окружение

Когда в программе написано `import tabulate`, Python ищет модуль в папках из списка `sys.path`, это было
на паре про модули. Сторонние библиотеки, которые ставят через `pip`, лежат в одной из этих папок,
обычно она называется `site-packages`. Посмотрим, какой Python выполняет этот ноутбук и где лежат его
библиотеки.

In [ ]:
import sys
from importlib.metadata import distributions

print(sys.executable)          # программа python, которая выполняет ноутбук
print([folder for folder in sys.path if folder.endswith("-packages")])
print("установлено библиотек:", len(list(distributions())))

/usr/bin/python3
['/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages']
установлено библиотек: 664


Пути у вас будут свои. Важно другое: папка с библиотеками у этого Python одна на все проекты. Пока
проект один, это удобно. Трудности начинаются, когда проектов несколько. Допустим, прошлогодний проект
написан под старую версию библиотеки, а новому нужна свежая. Обновите библиотеку для нового проекта —
может сломаться старый, а две версии одной библиотеки в одну папку не положить. Вторая трудность: через
полгода уже не вспомнить, какие из десятков установленных библиотек нужны проекту, а какие ставились
для чего-то другого.

Поэтому для проекта делают **виртуальное окружение** — отдельную папку со своим `site-packages`.
Интерпретатор и стандартную библиотеку окружение берёт из обычной установки Python, а сторонние
библиотеки у каждого окружения свои. Создаёт окружение модуль `venv` из стандартной библиотеки:
`python -m venv .venv`. Последнее слово — имя папки. Обычно её называют `.venv`: точка в начале
делает папку скрытой в macOS и Linux, а редакторы находят окружение с таким именем сами.

In [ ]:
games = '''
Аня Боря 1-0
Вика Аня 0-1
Боря Вика 1/2-1/2
Гоша Аня 1/2-1/2
Боря Гоша 1-0
Вика Гоша 0-1
'''

tournament_v1 = '''
"""Турнирная таблица по результатам партий из файла games.txt."""

from pathlib import Path

from tabulate import tabulate


def read_games(path: str) -> list[tuple[str, str, str]]:
    """Читает партии: в каждой строке белые, чёрные и результат."""
    games = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        white, black, result = line.split()
        games.append((white, black, result))
    return games


def count_points(games: list[tuple[str, str, str]]) -> dict[str, float]:
    """Очки игроков: за победу 1 очко, за поражение 0."""
    points: dict[str, float] = {}
    for white, black, result in games:
        points.setdefault(white, 0)
        points.setdefault(black, 0)
        if result == "1-0":
            points[white] += 1
        elif result == "0-1":
            points[black] += 1
    return points


def main() -> None:
    points = count_points(read_games("games.txt"))
    rows = sorted(points.items(), key=lambda item: (-item[1], item[0]))
    print(tabulate(rows, headers=["Игрок", "Очки"]))


if __name__ == "__main__":
    main()
'''

make_project({"games.txt": games, "tournament.py": tournament_v1})
run("-m", "venv", ".venv")
show_tree(depth=3)

$ python -m venv .venv
(в этой системе у venv нет модуля ensurepip, поэтому окружение создано пакетом virtualenv — результат тот же)
project/
    .venv/
        .gitignore
        CACHEDIR.TAG
        bin/
            activate
            activate.csh
            activate.fish
            activate.nu
            activate.ps1
            activate.xsh
            activate_this.py
            deactivate.csh
            pip
            pip-3.13
            pip3
            pip3.13
            python
            python3
            python3.13
        include/
        lib/
            python3.13/
        pyvenv.cfg
    games.txt
    tournament.py


Команда работает несколько секунд и ничего не печатает. В проекте появилась папка `.venv`.
В `bin` лежат программы окружения: `python`, `pip` и скрипты `activate` (в Windows эта папка называется
`Scripts`). В `lib` — своя папка `site-packages`, `lib64` в Linux — ссылка на ту же `lib`. Python 3.13
и новее кладёт в окружение ещё и файл `.gitignore`, зачем — станет понятно в блоке 6.

Посмотрим, что установлено в окружение, и заглянем в файл его настроек.

In [ ]:
venv_run("-m", "pip", "list")
show(".venv/pyvenv.cfg")

$ .venv/bin/python -m pip list
Package Version
------- -------
pip     26.3
----- .venv/pyvenv.cfg -----
  1  home = /usr/bin
  2  implementation = CPython
  3  python-version = 3.13
  4  version_info = 3.13.18.final.0
  5  version = 3.13.18
  6  executable = /usr/bin/python3.13
  7  command = /usr/bin/python3 -m virtualenv /tmp/py9_17_lecture/project/.venv
  8  virtualenv = 21.15.6
  9  include-system-site-packages = false
 10  base-prefix = /usr
 11  base-exec-prefix = /usr
 12  base-executable = /usr/bin/python3


В свежем окружении только `pip` (в Python 3.10 и 3.11 ещё `setuptools`), хотя в общем Python ноутбука
библиотек десятки. В `pyvenv.cfg` строка `home` хранит путь к папке обычного Python, из которой
окружение берёт интерпретатор и стандартную библиотеку, а `include-system-site-packages = false`
значит, что библиотеки общего Python окружению не видны.

**Что выведет?** `run` запускает Python ноутбука, `venv_run` — Python окружения.

In [ ]:
run("-c", "import IPython; print('IPython есть')")
venv_run("-c", "import IPython; print('IPython есть')")
venv_run("-c", "import json; print('json есть')")

```
$ python -c import IPython; print('IPython есть')
IPython есть
$ .venv/bin/python -c import IPython; print('IPython есть')
[ошибка] ModuleNotFoundError: No module named 'IPython'
$ .venv/bin/python -c import json; print('json есть')
json есть
```

Jupyter выполняет код ноутбука через IPython, значит в общем Python эта библиотека точно установлена.
Python окружения её не видит: он ищет сторонние библиотеки только в своём `site-packages`, а там пока
один `pip`. Модуль `json` из стандартной библиотеки находится, потому что стандартная библиотека
у окружения общая с обычным Python.

В терминале окружение обычно **активируют**: `source .venv/bin/activate` в macOS и Linux,
`.venv\Scripts\activate` в Windows. Активация меняет только то, какая программа запускается по слову
`python` или `pip`: после неё это `python` и `pip` из `.venv`, а в начале строки терминала появляется
`(.venv)`. Команда `deactivate` возвращает всё обратно. VS Code и PyCharm находят `.venv` в папке
проекта сами, нужно один раз выбрать его интерпретатором проекта. В ноутбуке активировать нечего,
поэтому `venv_run` запускает `.venv/bin/python` по полному пути. Результат тот же.

## 2. Зависимости проекта: `requirements.txt`

Турнирная таблица печатается с помощью библиотеки `tabulate`, которой в окружении нет. Запустим
программу Python'ом окружения.

In [ ]:
venv_run("tournament.py")

$ .venv/bin/python tournament.py
[ошибка] ModuleNotFoundError: No module named 'tabulate'


Поставим `tabulate` в окружение. Запись `tabulate==0.9.0` означает «ровно версию 0.9.0»; без номера
`pip` поставит самую свежую.

Обратите внимание, как запускается `pip`: `python -m pip`, то есть модуль `pip` того Python, в который
ставим библиотеку. Просто `pip` — это отдельная программа, и если Python на компьютере несколько,
легко получить `pip` от другого Python. Тогда библиотека «поставилась», а `import` её не находит.
Запись `python -m pip` от этой ошибки защищает.

In [ ]:
venv_run("-m", "pip", "install", "tabulate==0.9.0")
venv_run("-m", "pip", "list")
venv_run("tournament.py")

$ .venv/bin/python -m pip install tabulate==0.9.0
$ .venv/bin/python -m pip list
Package  Version
-------- -------
pip      26.3
tabulate 0.9.0
$ .venv/bin/python tournament.py
Игрок      Очки
-------  ------
Аня           2
Боря          1
Гоша          1
Вика          0


Теперь таблица печатается. Ничьи пока не учтены, исправим это во второй половине пары.

Зачем всё это, видно на примере. Пусть другой проект написан под старую версию `tabulate`. Сделаем
для него второе окружение `.venv-old` и поставим туда версию 0.8.10. Ключ `-q` просит `pip` не
печатать подробности.

In [ ]:
run("-m", "venv", ".venv-old")
venv_run("-m", "pip", "install", "-q", "tabulate==0.8.10", venv=".venv-old")
venv_run("-c", "import tabulate; print(tabulate.__version__)")
venv_run("-c", "import tabulate; print(tabulate.__version__)", venv=".venv-old")

$ python -m venv .venv-old
(в этой системе у venv нет модуля ensurepip, поэтому окружение создано пакетом virtualenv — результат тот же)
$ .venv-old/bin/python -m pip install -q tabulate==0.8.10
$ .venv/bin/python -c import tabulate; print(tabulate.__version__)
0.9.0
$ .venv-old/bin/python -c import tabulate; print(tabulate.__version__)
0.8.10


Две версии одной библиотеки живут на одном компьютере и друг другу не мешают: у каждого окружения
свой `site-packages`.

Теперь главное: как повторить окружение на другом компьютере. Для этого список библиотек проекта
с точными версиями записывают в файл **`requirements.txt`** в корне проекта. Он уже был в структуре
проекта на паре 15. Получить такой список помогает команда `pip freeze`: она печатает все библиотеки
окружения в виде `имя==версия`.

In [ ]:
venv_run("-m", "pip", "freeze")

$ .venv/bin/python -m pip freeze
tabulate==0.9.0


В терминале вывод сразу направляют в файл: `python -m pip freeze > requirements.txt`. Мы запишем файл
через `write`.

Представим, что проект скачали на другой компьютер. Папки `.venv` там нет, её не пересылают
(почему — ниже). Есть код и `requirements.txt`. Удалим оба окружения и восстановим нужное по списку.
`pip install -r requirements.txt` ставит всё, что перечислено в файле, с указанными версиями.

In [ ]:
write("requirements.txt", "tabulate==0.9.0\n")
remove(".venv")
remove(".venv-old")

run("-m", "venv", ".venv")
venv_run("-m", "pip", "install", "-q", "-r", "requirements.txt")
venv_run("tournament.py")

$ python -m venv .venv
(в этой системе у venv нет модуля ensurepip, поэтому окружение создано пакетом virtualenv — результат тот же)
$ .venv/bin/python -m pip install -q -r requirements.txt
$ .venv/bin/python tournament.py
Игрок      Очки
-------  ------
Аня           2
Боря          1
Гоша          1
Вика          0


Несколько правил про зависимости.

`freeze` запускают в окружении проекта, а не в общем Python. В начале пары мы посчитали библиотеки
общего Python ноутбука: их десятки, а в Colab сотни, и почти все не нужны проекту. В окружении
проекта стоит только нужное, поэтому и список получается честный.

Версии фиксируют точно, через `==`. Без версии через полгода `pip` поставит другую, и программа может
повести себя иначе. Пример был на прошлой паре: в новой версии ruff поменялся набор правил
по умолчанию, поэтому мы писали правила явно. Если в `requirements.txt` проекта записан `ruff`
с точной версией, у всех в классе одни и те же замечания.

В репозиторий кладут `requirements.txt`, а `.venv` — нет. Окружение весит десятки мегабайт, хранит
абсолютные пути (строка `home` в `pyvenv.cfg`) и привязано к системе: окружение, сделанное в Windows,
не заработает в macOS. Зато его всегда можно пересоздать по `requirements.txt`, поэтому удалять
`.venv` не страшно.

В Colab окружения обычно не делают: каждая сессия — новая машина, и после перезапуска всё
установленное пропадает. Там первой ячейкой ставят зависимости: `%pip install -r requirements.txt`
или `%pip install tabulate==0.9.0`.

## 3. Git: репозиторий и первый коммит

Как хранят версии проекта без специальных программ: копируют папку — `турнир`, `турнир_новый`,
`турнир_финал`, `турнир_финал_2` — и пересылают архивы. Через неделю уже не понять, чем копии
отличаются, какая из них последняя и что поменял одноклассник.

**Git** — система контроля версий: программа, которая хранит историю проекта как цепочку снимков.
Снимок называется **коммитом**. В коммите записано состояние всех файлов проекта, а ещё кто, когда
и зачем его сделал. Git написал Линус Торвальдс в 2005 году для разработки ядра Linux, сейчас им
пользуется почти каждый программист.

Сначала Git нужно представиться: каждый коммит подписан именем и почтой автора. Это делают один раз
на компьютере, ключ `--global` значит «для всех проектов».

In [ ]:
git('config --global user.name "Аня Смирнова"')
git('config --global user.email "anya@example.com"')
git("config --global --list")

$ git config --global user.name "Аня Смирнова"
$ git config --global user.email "anya@example.com"
$ git config --global --list
user.name=Аня Смирнова
user.email=anya@example.com


Служебная ячейка подменила файл общих настроек Git временным, поэтому ваши настоящие настройки
не изменились. На своём компьютере выполните эти две команды со своим именем.

Создадим проект заново, без окружений, и превратим его папку в **репозиторий** командой `git init`.

In [ ]:
readme = '''
# Шахматный турнир

Печатает турнирную таблицу по результатам партий из `games.txt`.
'''

make_project({
    "tournament.py": tournament_v1,
    "games.txt": games,
    "README.md": readme,
    "requirements.txt": "tabulate==0.9.0\n",
})
git("init -b main")
show_tree()

$ git init -b main
Initialized empty Git repository in /tmp/py9_17_lecture/project/.git/
project/
    .git/
    README.md
    games.txt
    requirements.txt
    tournament.py


`git init` создал в проекте скрытую папку `.git`: в ней Git хранит всю историю. Если её удалить,
история пропадёт, а файлы проекта останутся. Ключ `-b main` называет основную ветку `main`. О ветках
поговорим на следующей паре, а без ключа основная ветка может называться `master`.

Главная команда Git — `git status`: что изменилось и что Git собирается делать.

In [ ]:
git("status")

$ git status
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md
	games.txt
	requirements.txt
	tournament.py

nothing added to commit but untracked files present (use "git add" to track)


Git видит четыре файла, но они **неотслеживаемые** (untracked): Git их ни разу не сохранял.
Чтобы понять, как файлы попадают в коммит, нужна схема из трёх мест.

```
рабочая папка  --- git add --->  индекс  --- git commit --->  история (коммиты)
```

**Рабочая папка** — файлы, которые вы видите и редактируете. **Индекс** (по-английски staging area) —
черновик следующего коммита: командой `git add` в него кладут то, что должно войти в коммит.
**Коммит** записывает в историю ровно то, что лежит в индексе. Промежуточный шаг нужен потому, что
в рабочей папке часто лежат изменения разного смысла: исправление ошибки, новая функция, черновик.
Через индекс коммит собирают только из того, что относится к одному делу.

Положим в индекс программу и данные, а описание и зависимости пока оставим.

In [ ]:
git("add tournament.py games.txt")
git("status")

$ git add tournament.py games.txt
$ git status
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   games.txt
	new file:   tournament.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md
	requirements.txt


Теперь два файла в разделе «Changes to be committed» — они войдут в коммит, остальные два по-прежнему
неотслеживаемые. Делаем коммит.

In [ ]:
git('commit -m "Подсчёт очков турнира"')

$ git commit -m "Подсчёт очков турнира"
[main (root-commit) 97a3490] Подсчёт очков турнира
 2 files changed, 43 insertions(+)
 create mode 100644 games.txt
 create mode 100644 tournament.py


Ключ `-m` задаёт **сообщение коммита** — короткое описание изменения. Без `-m` Git откроет текстовый
редактор, чтобы сообщение написали там.

В первой строке ответа — ветка, номер коммита и сообщение. Номер — это **хеш** из 40 шестнадцатеричных
цифр. Git вычисляет его по содержимому файлов, автору, времени и предыдущему коммиту, обычно хватает
первых семи цифр. Измените хотя бы один символ — получится другой хеш. В этом ноутбуке хеши у всех
в классе получатся одинаковыми: имя и почту мы задали одни и те же, а время служебная ячейка
подставляет сама. В обычной работе хеш у каждого коммита свой.

Закоммитим остальные файлы и посмотрим историю.

In [ ]:
git("add README.md requirements.txt")
git('commit -m "Описание проекта и зависимости"')
git("log")

$ git add README.md requirements.txt
$ git commit -m "Описание проекта и зависимости"
[main f4a4d5b] Описание проекта и зависимости
 2 files changed, 4 insertions(+)
 create mode 100644 README.md
 create mode 100644 requirements.txt
$ git log
commit f4a4d5bdfba78a96b39f82bb8463b0032147df8f
Author: Аня Смирнова <anya@example.com>
Date:   Mon Nov 9 10:05:00 2026 +0300

    Описание проекта и зависимости

commit 97a34907d5f848b1bca19031d5d6b8a9d999fa78
Author: Аня Смирнова <anya@example.com>
Date:   Mon Nov 9 10:00:00 2026 +0300

    Подсчёт очков турнира


`git log` показывает историю от новых коммитов к старым: хеш, автор, дата, сообщение. Короткий
вариант — `git log --oneline`, по строке на коммит.

## 4. Изменения: `status`, `diff` и индекс

В турнире пока не считаются ничьи: результат `1/2-1/2` не приносит очков никому. Исправим
`count_points` и посмотрим, как Git видит изменение.

In [ ]:
tournament_v2 = tournament_v1.replace(
    "за победу 1 очко, за поражение 0.",
    "за победу 1 очко, за ничью пол-очка, за поражение 0.",
).replace(
    "            points[black] += 1\n",
    "            points[black] += 1\n"
    "        else:\n"
    "            points[white] += 0.5\n"
    "            points[black] += 0.5\n",
)
write("tournament.py", tournament_v2)
git("status")
git("diff")

`git status` показывает файл в разделе «Changes not staged for commit»: он изменён, но в индекс
изменение не положено. `git diff` показывает, что именно изменилось. Строки с `-` удалены, строки
с `+` добавлены, строки без знака — соседний текст для ориентира. Изменений два, и каждое показано
отдельным фрагментом. Заголовок фрагмента говорит, где он: `@@ -24,6 +24,9 @@` значит «в старой версии
шесть строк начиная с 24-й, в новой на их месте девять». Изменённая строка, как докстрока в первом
фрагменте, выглядит как удалённая старая и добавленная новая.

Положим изменение в индекс и снова сравним.

In [ ]:
git("add tournament.py")
git("diff")
git("diff --staged")

Первая команда ничего не напечатала. `git diff` сравнивает рабочую папку с индексом, а после `add`
они совпадают. `git diff --staged` сравнивает индекс с последним коммитом и показывает, что войдёт
в следующий коммит. Перед каждым коммитом полезно посмотреть `git status` и `git diff --staged`.

In [ ]:
git('commit -m "Учитывать ничьи"')

**Что выведет?** Допишем в README, как запускать программу, положим файл в индекс, потом допишем ещё
одну строку и сделаем коммит. `git status --short` печатает состояние коротко: по строке на файл.
`git show HEAD:README.md` печатает файл в том виде, в каком он записан в последнем коммите.

In [ ]:
write("README.md", readme + "\nЗапуск: `python tournament.py`.\n")
git("add README.md")
write("README.md", readme + "\nЗапуск: `python tournament.py`.\nЗа ничью — пол-очка каждому.\n")
git('commit -m "Как запускать программу"')
git("status --short")
git("show HEAD:README.md")

```
$ git status --short
 M README.md
$ git show HEAD:README.md
# Шахматный турнир

Печатает турнирную таблицу по результатам партий из `games.txt`.

Запуск: `python tournament.py`.
```

Строки про ничью в коммите нет. `git add` кладёт в индекс содержимое файла **в тот момент**, а не
«файл вообще». Коммит записал то, что лежало в индексе, — версию до второй правки. После коммита
рабочая папка снова отличается от последнего коммита, поэтому `git status --short` показывает
` M README.md`. Буква `M` во второй колонке значит «изменён в рабочей папке, в индекс не положен»,
в первой колонке было бы «изменён и положен в индекс».

Отсюда частая ошибка: поправили файл, сделали `add`, поправили ещё раз и закоммитили — последняя
правка в коммит не попала. Перед коммитом смотрите `git status`. Положим и вторую правку.

In [ ]:
git("add README.md")
git('commit -m "Правило про ничьи в README"')
git("log --oneline")

**Каким должен быть коммит.** Один коммит — одно законченное изменение: исправили ошибку — коммит,
добавили функцию — ещё один коммит. Такой коммит легко понять, проверить и при необходимости отменить,
не задев остальное. Сообщение отвечает на вопрос «что сделает этот коммит?»: «Учитывать ничьи»,
«Сортировать игроков с равными очками по имени». Первую строку делают короткой, до 50–70 символов.
Если нужно объяснить, почему сделано именно так, подробности пишут ниже после пустой строки.

| плохо | лучше |
|---|---|
| `fix` | `Учитывать ничьи при подсчёте очков` |
| `изменения` | `Сортировать игроков с равными очками по имени` |
| `всё сделал` (три изменения в одном коммите) | три коммита, по одному на изменение |
| `asdf` | `Описание проекта и зависимости` |

Команда `git add .` кладёт в индекс всё, что изменилось в папке, вместе со случайными файлами.
Ей удобно пользоваться, но перед коммитом обязательно проверьте `git status`. Как сделать, чтобы
лишние файлы вообще не попадали в `git add .`, — в блоке 6.

## 5. История и отмена изменений

У последнего коммита текущей ветки есть имя **`HEAD`**: «где мы сейчас». `HEAD~1` — коммит перед ним,
`HEAD~2` — ещё на шаг раньше. Вместо этих имён можно писать хеш коммита или его первые цифры.
`git show` показывает один коммит: сообщение и изменения.

In [ ]:
git("show HEAD~2")

Если указать после `--` имя файла, `git log` покажет только коммиты, которые меняли этот файл.

In [ ]:
git("log --oneline -- tournament.py")

История нужна не только чтобы смотреть, но и чтобы исправлять ошибки. Разберём три случая.

**Испортили файл и ещё не делали `add`.** Например, случайно стёрли половину кода. `git restore файл`
вернёт файл к версии из индекса, а если в индекс ничего не клали — к версии из последнего коммита.
Осторожно: несохранённые в Git правки этого файла пропадут насовсем, Git о них ничего не знал.

In [ ]:
write("tournament.py", "# здесь был код\n")
git("status --short")
git("restore tournament.py")
git("status --short")

После `restore` команда `git status --short` ничего не печатает: файл снова совпадает с последним
коммитом.

**Положили в индекс лишнее.** `git restore --staged файл` убирает файл из индекса, а в рабочей папке
оставляет его как есть.

In [ ]:
write("notes.txt", "Не забыть: следующий тур в субботу\n")
git("add .")
git("status --short")
git("restore --staged notes.txt")
git("status --short")
remove("notes.txt")

`A` в первой колонке — «новый файл, положен в индекс», `??` — «неотслеживаемый».

**Неудачный коммит уже в истории.** Допустим, кто-то «упростил» подсчёт очков, потерял ничьи
и закоммитил. Ключ `-a` у `git commit` сам кладёт в индекс все изменённые отслеживаемые файлы,
`-am` — это `-a` и `-m` вместе. `git revert коммит` создаёт **новый коммит**, который отменяет
изменения указанного.

In [ ]:
write("tournament.py", tournament_v1)
git('commit -am "Упростить подсчёт очков"')
git("revert --no-edit HEAD")
git("log --oneline")
git("diff HEAD~2 HEAD")

Неудачный коммит остался в истории, а над ним появился коммит с его отменой. `git diff HEAD~2 HEAD`
ничего не напечатал: проект снова такой, каким был до неудачного коммита. Ключ `--no-edit` берёт
стандартное сообщение «Revert …», без него Git откроет редактор.

История при этом не переписывается, и это безопасно, даже если её уже видели другие. Есть и команда
`git reset --hard коммит`: она «перематывает» ветку назад, выбрасывает коммиты после указанного
и заодно все незакоммиченные правки. Пользоваться ею можно, только понимая, что делаете, и никогда —
для коммитов, которые уже отправлены другим. Об этом поговорим, когда дойдём до удалённых репозиториев.

| что случилось | что делать |
|---|---|
| испортили файл, `add` не делали | `git restore файл` |
| положили в индекс лишнее | `git restore --staged файл` |
| нужна версия файла из старого коммита | `git restore --source=HEAD~2 файл`, потом коммит |
| неудачный коммит уже в истории | `git revert коммит` |
| хочется посмотреть, что было в коммите | `git show коммит` |

## 6. Что не хранить в репозитории: `.gitignore`

Вернём в проект окружение, заставим Python создать кеш байт-кода (модуль `py_compile` делает это явно)
и положим рядом файл с ключом доступа к сайту, куда программа будет отправлять результаты. Что
скажет Git?

In [ ]:
run("-m", "venv", ".venv")
run("-m", "py_compile", "tournament.py")
write("token.txt", "секретный-ключ-для-сайта-турниров\n")
git("status --short")

Git предлагает сохранить и кеш, и ключ. Если у вас Python 3.10–3.12, в списке будет ещё `.venv/`.
В Python 3.13 `venv` сам кладёт внутрь окружения файл `.gitignore` с одной строкой `*` («игнорировать
всё»), и окружение Git уже не видит. Рассчитывать на это нельзя: у одноклассника может быть другая
версия Python или окружение, созданное другой программой.

В репозитории не место:

- окружению `.venv` — его пересоздают по `requirements.txt`;
- кешу байт-кода `__pycache__` — Python создаёт его сам;
- служебным файлам редакторов и Jupyter: `.idea`, `.vscode`, `.ipynb_checkpoints`;
- большим файлам данных и всему, что программа генерирует сама;
- паролям и ключам доступа.

С ключами особенно строго. Если ключ попал в коммит, он остаётся в истории, даже если потом удалить
файл, и его видит каждый, у кого есть копия репозитория. Такой ключ считают украденным и заменяют.

Чтобы Git не замечал лишнего, имена и шаблоны имён записывают в файл **`.gitignore`** в корне проекта.

In [ ]:
write(".gitignore", '''
    # окружение и кеш Python
    .venv/
    __pycache__/

    # ключи доступа
    token.txt
''')
git("status --short")
git("check-ignore -v token.txt .venv/pyvenv.cfg")

Теперь Git видит только сам `.gitignore`. Команда `git check-ignore -v` объясняет, почему файл
игнорируется: в каком файле и в какой строке подходящий шаблон.

| строка в `.gitignore` | что игнорируется |
|---|---|
| `token.txt` | файлы и папки с таким именем в любом месте проекта |
| `__pycache__/` | только папки с таким именем, в любом месте, вместе с содержимым |
| `/build` | только `build` в корне проекта |
| `data/*.csv` | файлы `.csv` прямо в папке `data` в корне проекта |
| `*.log` | все файлы с окончанием `.log` |
| `!important.log` | исключение из правил выше: этот файл не игнорировать |
| `# текст` | комментарий |

`.gitignore` действует только на файлы, которых ещё нет в репозитории. Если файл уже закоммичен,
строка в `.gitignore` его не уберёт. Сначала выполняют `git rm --cached файл` (убрать из индекса,
но оставить на диске), потом делают коммит.

Сам `.gitignore` кладут в репозиторий: правила нужны всем, кто работает с проектом.

In [ ]:
git("add .")
git("status --short")
git('commit -m "Не хранить окружение, кеш и ключи"')
git("log --oneline")

## Итоги

- Виртуальное окружение — папка `.venv` со своими библиотеками. Создаётся командой
  `python -m venv .venv`. Интерпретатор и стандартная библиотека общие с обычным Python, сторонние
  библиотеки у каждого окружения свои, поэтому разные проекты могут жить на разных версиях.
- Активация (`source .venv/bin/activate`, в Windows `.venv\Scripts\activate`) меняет только то, какие
  `python` и `pip` запускаются в терминале. Редакторы находят `.venv` сами.
- Библиотеку ставят командой `python -m pip install имя==версия` — тем Python, которому она нужна.
- `requirements.txt` — список зависимостей с точными версиями. Получают его `pip freeze` в окружении
  проекта, восстанавливают окружение командой `pip install -r requirements.txt`. В репозиторий кладут
  `requirements.txt`, а не `.venv`.
- Git хранит историю проекта как цепочку коммитов. Коммит — снимок всех файлов плюс автор, время
  и сообщение. Номер коммита — хеш, он зависит от всего этого и от предыдущего коммита.
- Три места: рабочая папка, индекс, история. `git add` кладёт в индекс содержимое файла в момент
  команды, `git commit` записывает в историю то, что лежит в индексе.
- `git status` — что изменилось и где; `git diff` — рабочая папка против индекса; `git diff --staged` —
  индекс против последнего коммита; `git log` и `git show` — история.
- Один коммит — одно изменение. Сообщение говорит, что делает коммит.
- Отмена: `git restore файл`, `git restore --staged файл`, `git restore --source=коммит файл`,
  `git revert коммит`. `git reset --hard` переписывает историю и выбрасывает правки.
- В `.gitignore` записывают окружение, `__pycache__`, служебные файлы, большие данные, ключи и пароли.

**Что дальше.** На следующей паре — ветки: как вести несколько изменений параллельно, сливать их
и разрешать конфликты. Потом — удалённый репозиторий, совместная работа и обзор кода.

**Где почитать** (по желанию):

- [Pro Git](https://git-scm.com/book/ru/v2) — книга о Git на русском; главы 1 и 2 — ровно то, что было
  сегодня;
- [Learn Git Branching](https://learngitbranching.js.org/?locale=ru_RU) — тренажёр в браузере, с ним
  удобно готовиться к паре про ветки;
- [venv — документация Python](https://docs.python.org/3/library/venv.html) — всё о виртуальных
  окружениях;
- [Install packages in a virtual environment using pip and venv](https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/) —
  руководство по `pip` и окружениям.